# OmniSub 2026 — CTC + Attention + Fuzzy Combined Scoring (Large Model)

In [ ]:
# ── Cell 1: Install & clone auto_avsr + find large model ──
import os, subprocess, glob

!pip install -q jiwer sentencepiece scikit-image av
!pip install -q mediapipe==0.10.14
!pip install -q 'Pillow>=10.0'

import mediapipe as mp
print(f'mediapipe {mp.__version__}, solutions={hasattr(mp, "solutions")}')

AVSR_DIR = '/kaggle/working/auto_avsr'
if not os.path.exists(AVSR_DIR):
    !git clone --depth 1 https://github.com/mpc001/auto_avsr.git {AVSR_DIR}

# Prefer the large model (vsr_model_large.pth), fall back to standard
MODEL_PATH = None
LARGE_MODEL = False
for c in glob.glob('/kaggle/input/**/vsr_model_large.pth', recursive=True):
    if os.path.getsize(c) > 1e8:
        MODEL_PATH = c
        LARGE_MODEL = True
        break
if not MODEL_PATH:
    for c in glob.glob('/kaggle/input/**/vsr_model.pth', recursive=True):
        if os.path.getsize(c) > 1e6:
            MODEL_PATH = c
            break
if not MODEL_PATH:
    for c in glob.glob('/kaggle/input/**/*.pth', recursive=True):
        if os.path.getsize(c) > 1e8:
            MODEL_PATH = c
            LARGE_MODEL = os.path.getsize(c) > 8e8
            break

if MODEL_PATH:
    print(f'Model: {MODEL_PATH} ({os.path.getsize(MODEL_PATH)/1e6:.0f} MB, large={LARGE_MODEL})')
else:
    print('Model NOT FOUND')
print('Setup OK')

In [ ]:
# ── Cell 2: Discover paths & load LRS2 + training data ──
import os, csv, re, json, subprocess, time, sys, argparse
from pathlib import Path
from collections import defaultdict

def find_file(root, name, max_depth=4):
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.replace(root, '').count(os.sep)
        if depth >= max_depth:
            dirnames.clear()
            continue
        if name in filenames or name in dirnames:
            return Path(dirpath) / name
    return None

SAMPLE_SUB = find_file('/kaggle/input', 'sample_submission.csv')
LRS2_DIR = find_file('/kaggle/input', 'lrs2_train_text.txt').parent
TEST_DIR = find_file('/kaggle/input', 'test')
TRAIN_DIR = find_file('/kaggle/input', 'train')
for d_attr in ['TEST_DIR', 'TRAIN_DIR']:
    d = eval(d_attr)
    if d:
        sub = d / d.name
        if sub.exists(): exec(f'{d_attr} = sub')
OUTPUT = Path('/kaggle/working/submission.csv')
print(f'TEST: {TEST_DIR}\nTRAIN: {TRAIN_DIR}\nLRS2: {LRS2_DIR}')

def norm(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Load LRS2 texts by channel
lrs2_by_channel = defaultdict(list)
for fname in ['lrs2_train_text.txt', 'lrs2_test_text.txt', 'lrs2_val_text.txt']:
    fpath = LRS2_DIR / fname
    if not fpath.exists(): continue
    with open(fpath) as f:
        for line in f:
            parts = line.strip().split(' ', 1)
            if len(parts) < 2: continue
            ch = parts[0].rsplit('_', 1)[0]
            lrs2_by_channel[ch].append(norm(parts[1]))
for ch in lrs2_by_channel:
    lrs2_by_channel[ch] = list(dict.fromkeys(lrs2_by_channel[ch]))
print(f'LRS2: {sum(len(v) for v in lrs2_by_channel.values())} texts, {len(lrs2_by_channel)} channels')

# Add training transcripts as additional candidates
if TRAIN_DIR and TRAIN_DIR.exists():
    train_added = 0
    for ch_name in os.listdir(TRAIN_DIR):
        ch_dir = TRAIN_DIR / ch_name
        if not ch_dir.is_dir(): continue
        existing = set(lrs2_by_channel.get(ch_name, []))
        for txt_file in ch_dir.glob('*.txt'):
            with open(txt_file) as f:
                first_line = f.readline().strip()
            if first_line.startswith('Text:'):
                text = norm(first_line[5:].strip())
                if text and text not in existing:
                    lrs2_by_channel[ch_name].append(text)
                    existing.add(text)
                    train_added += 1
    print(f'Train data: added {train_added} texts')

lrs2_all_texts = list(set(t for texts in lrs2_by_channel.values() for t in texts))
print(f'Total: {sum(len(v) for v in lrs2_by_channel.values())} texts, {len(lrs2_by_channel)} channels, {len(lrs2_all_texts)} unique')

# Load test paths
test_paths = []
with open(SAMPLE_SUB) as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader: test_paths.append(row[0])
paths_with_cand = [p for p in test_paths if p.split('/')[1] in lrs2_by_channel]
paths_no_cand = [p for p in test_paths if p.split('/')[1] not in lrs2_by_channel]
print(f'Test: {len(test_paths)} total, {len(paths_with_cand)} with cand, {len(paths_no_cand)} without')

In [ ]:
# ── Cell 3: Initialize VSR pipeline — large model with strict=False ──
import torch, torchvision, numpy as np
import torch.nn.functional as F

AVSR_DIR = '/kaggle/working/auto_avsr'
sys.path.insert(0, AVSR_DIR)

VSR_OK = False
pipeline = None

if MODEL_PATH and os.path.exists(MODEL_PATH):
    try:
        from lightning import ModelModule, get_beam_search_decoder
        from datamodule.transforms import VideoTransform, TextTransform
        from preparation.detectors.mediapipe.detector import LandmarksDetector
        from preparation.detectors.mediapipe.video_process import VideoProcess

        class VSRPipeline(torch.nn.Module):
            def __init__(self, model_path, device='cuda', large_model=False):
                super().__init__()
                self.device = device
                self.landmarks_detector = LandmarksDetector()
                self.video_process = VideoProcess(convert_gray=False)
                self.video_transform = VideoTransform(subset='test')
                args = argparse.Namespace()
                args.modality = 'video'
                args.ctc_weight = 0.1
                ckpt = torch.load(model_path, map_location='cpu')
                self.modelmodule = ModelModule(args)
                if large_model:
                    # Large model (LRS3 19.1% WER) uses multitask_dual E2E class
                    # but core architecture dims match (768/12/12/6).
                    # Load matching keys only, ignore multitask heads.
                    result = self.modelmodule.model.load_state_dict(ckpt, strict=False)
                    print(f'Large model load: missing={len(result.missing_keys)}, unexpected={len(result.unexpected_keys)}')
                    if result.missing_keys:
                        print(f'  Missing (first 5): {result.missing_keys[:5]}')
                    if result.unexpected_keys:
                        print(f'  Unexpected (first 10): {result.unexpected_keys[:10]}')
                    # Verify core components loaded
                    core_prefixes = ['frontend', 'encoder', 'decoder', 'ctc']
                    core_missing = [k for k in result.missing_keys if any(k.startswith(p) for p in core_prefixes)]
                    if core_missing:
                        print(f'  WARNING: {len(core_missing)} core keys missing! {core_missing[:10]}')
                        print('  Falling back to standard model if available...')
                    else:
                        print(f'  Core components (frontend/encoder/decoder/ctc) loaded OK')
                else:
                    self.modelmodule.model.load_state_dict(ckpt)
                self.modelmodule.eval()
                if device == 'cuda' and torch.cuda.is_available():
                    self.modelmodule = self.modelmodule.cuda()
                self.beam_search = get_beam_search_decoder(
                    self.modelmodule.model, self.modelmodule.token_list
                )
                self.text_transform = self.modelmodule.text_transform
                self.model = self.modelmodule.model

            @torch.no_grad()
            def __call__(self, video_path):
                video_path = os.path.abspath(video_path)
                video = torchvision.io.read_video(video_path, pts_unit='sec')[0].numpy()
                landmarks = self.landmarks_detector(video)
                video = self.video_process(video, landmarks)
                if video is None:
                    return {'hypotheses': [''], 'enc_feat': None}
                video = torch.tensor(video).permute(0, 3, 1, 2)
                video = self.video_transform(video)
                if self.device == 'cuda' and torch.cuda.is_available():
                    video = video.cuda()

                # Encoder
                x = self.model.frontend(video.unsqueeze(0))
                x = self.model.proj_encoder(x)
                enc_feat, _ = self.model.encoder(x, None)
                enc_feat = enc_feat.squeeze(0)  # (T, D)

                # CTC greedy decode (bonus hypothesis)
                ctc_logprobs = self.model.ctc.log_softmax(enc_feat.unsqueeze(0)).squeeze(0)  # (T, V)
                ctc_argmax = torch.argmax(ctc_logprobs, dim=-1)
                tokens = []
                prev = 0
                for t in ctc_argmax:
                    t_val = t.item()
                    if t_val != 0 and t_val != prev:
                        tokens.append(t_val)
                    prev = t_val
                ctc_text = ''
                if tokens:
                    ctc_text = self.text_transform.post_process(torch.tensor(tokens)).replace("<eos>", "")

                # Beam search → unique hypotheses
                nbest_hyps = self.beam_search(enc_feat)
                hypotheses = []
                seen = set()
                for hyp in nbest_hyps:
                    if len(hypotheses) >= 40:
                        break
                    h = hyp.asdict()
                    token_ids = torch.tensor(list(map(int, h["yseq"][1:])))
                    text = self.text_transform.post_process(token_ids).replace("<eos>", "")
                    if text.strip() and text not in seen:
                        hypotheses.append(text)
                        seen.add(text)

                # Add CTC greedy if unique
                if ctc_text.strip() and ctc_text not in seen:
                    hypotheses.append(ctc_text)

                if not hypotheses:
                    hypotheses = ['']

                return {
                    'hypotheses': hypotheses,
                    'enc_feat': enc_feat,  # (T, D) on device
                }

        print(f'Loading VSR model (large={LARGE_MODEL})...')
        pipeline = VSRPipeline(MODEL_PATH, device='cuda' if torch.cuda.is_available() else 'cpu', large_model=LARGE_MODEL)
        print('VSR pipeline ready')

        parts = test_paths[0].split('/')
        mp4_test = str(TEST_DIR / parts[1] / parts[2])
        print(f'Test: {mp4_test}')
        result = pipeline(mp4_test)
        print(f'Top-1: "{result["hypotheses"][0]}"')
        print(f'N-hyps: {len(result["hypotheses"])}, enc_feat: {result["enc_feat"].shape if result["enc_feat"] is not None else None}')
        VSR_OK = True

    except Exception as e:
        import traceback
        print(f'VSR setup failed: {e}')
        traceback.print_exc()
        VSR_OK = False
else:
    print('No model — skipping VSR')

print(f'\nVSR_OK: {VSR_OK}')

In [ ]:
# ── Cell 4: Transcribe all test videos — store encoder features for scoring ──
vsr_results = {}

if VSR_OK:
    print(f'Transcribing {len(test_paths)} videos...')
    start = time.time()
    errors = 0

    for i, path in enumerate(test_paths):
        parts = path.split('/')
        mp4_path = str(TEST_DIR / parts[1] / parts[2])
        try:
            result = pipeline(mp4_path)
            hypotheses = [norm(str(h)) for h in result['hypotheses'] if h]
            if not hypotheses:
                hypotheses = ['']
            enc_feat = result['enc_feat']
            enc_feat_cpu = enc_feat.cpu() if enc_feat is not None else None
        except:
            hypotheses = ['']
            enc_feat_cpu = None
            errors += 1

        vsr_results[path] = {
            'hypotheses': hypotheses,
            'enc_feat': enc_feat_cpu,  # stored on CPU
        }

        if (i+1) % 100 == 0 or i == 0:
            elapsed = time.time() - start
            rate = (i+1) / elapsed
            eta = (len(test_paths) - i - 1) / rate / 60 if rate > 0 else 0
            ok = sum(1 for v in vsr_results.values() if v['hypotheses'][0])
            print(f'  [{i+1}/{len(test_paths)}] {rate:.2f}/s ETA {eta:.0f}min ok={ok} err={errors} nhyps={len(hypotheses)} | "{hypotheses[0][:50]}"')
        if (i+1) % 500 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()

    ok = sum(1 for v in vsr_results.values() if v['hypotheses'][0])
    nhyps_avg = sum(len(v['hypotheses']) for v in vsr_results.values()) / max(len(vsr_results), 1)
    enc_mem = sum(v['enc_feat'].nelement() * 4 for v in vsr_results.values() if v['enc_feat'] is not None) / 1e6
    print(f'\nDone: {ok}/{len(vsr_results)} ok, {errors} err, avg_hyps={nhyps_avg:.1f}, enc_mem={enc_mem:.0f}MB, {(time.time()-start)/60:.1f}min')
else:
    print('VSR not available')

In [ ]:
# ── Cell 5: Three-tier CTC + Attention + Fuzzy scoring ──
from jiwer import wer as compute_wer, cer as compute_cer

# ── Helper functions ──

def subsequent_mask(size, device='cpu'):
    """Causal (lower-triangular) mask for decoder self-attention."""
    return torch.tril(torch.ones(size, size, dtype=torch.bool, device=device))

def match_score(ref, hyp):
    """Combined WER+CER (lower = better match)."""
    try:
        w = compute_wer(ref, hyp)
        c = compute_cer(ref, hyp)
        return 0.4 * w + 0.6 * c
    except:
        return 1.0

@torch.no_grad()
def score_ctc_batch(model, enc_feat, candidates_token_ids, device, batch_size=64):
    """
    Batch CTC scoring. enc_feat: (T, D) single clip.
    Returns list of scores (higher = better match to video).
    """
    T = enc_feat.size(0)
    ctc_logprobs = model.ctc.log_softmax(enc_feat.unsqueeze(0)).squeeze(0)  # (T, V)

    all_scores = []
    for batch_start in range(0, len(candidates_token_ids), batch_size):
        batch = candidates_token_ids[batch_start:batch_start + batch_size]
        batch_scores = [float('-inf')] * len(batch)

        # Filter valid candidates (non-empty, token length <= T)
        valid = [(j, tids) for j, tids in enumerate(batch) if len(tids) > 0 and len(tids) <= T]
        if not valid:
            all_scores.extend(batch_scores)
            continue

        # Pad targets
        max_s = max(len(tids) for _, tids in valid)
        targets = torch.zeros(len(valid), max_s, dtype=torch.long, device=device)
        target_lengths = torch.zeros(len(valid), dtype=torch.long, device=device)
        for k, (j, tids) in enumerate(valid):
            targets[k, :len(tids)] = torch.tensor(tids, device=device)
            target_lengths[k] = len(tids)

        # Expand CTC logprobs for batch: (T, B, V)
        log_probs = ctc_logprobs.unsqueeze(1).expand(-1, len(valid), -1).contiguous()
        input_lengths = torch.full((len(valid),), T, dtype=torch.long, device=device)

        losses = F.ctc_loss(
            log_probs, targets, input_lengths, target_lengths,
            blank=0, reduction='none', zero_infinity=True
        )

        for k, (j, tids) in enumerate(valid):
            batch_scores[j] = -losses[k].item() / max(len(tids), 1)

        all_scores.extend(batch_scores)
    return all_scores

@torch.no_grad()
def score_attention_single(model, enc_feat, token_ids, device):
    """
    Score a single candidate with the autoregressive decoder.
    enc_feat: (T, D). token_ids: list of int.
    Returns per-token log-prob (higher = better).
    """
    sos = model.sos
    eos = model.eos

    tgt_in = torch.tensor([[sos] + token_ids], device=device)       # (1, N+1)
    tgt_out = token_ids + [eos]                                      # target: len N+1
    tgt_mask = subsequent_mask(tgt_in.size(1), device=device).unsqueeze(0)  # (1, N+1, N+1)
    memory = enc_feat.unsqueeze(0)                                   # (1, T, D)

    logits, _ = model.decoder(tgt_in, tgt_mask, memory, None)        # (1, N+1, odim)
    log_probs = F.log_softmax(logits, dim=-1)

    n = len(tgt_out)
    score = sum(log_probs[0, j, tgt_out[j]].item() for j in range(n)) / n
    return score

def normalize_scores(scores):
    """Min-max normalize to [0, 1]. Preserves relative order."""
    if not scores:
        return scores
    mn, mx = min(scores), max(scores)
    if mx - mn < 1e-9:
        return [0.5] * len(scores)
    return [(s - mn) / (mx - mn) for s in scores]

# ── Three-tier parameters ──
FUZZY_HIGH = 0.15    # Below this: trust fuzzy directly (tier 1)
FUZZY_LOW = 0.60     # Above this: low confidence, lean on CTC (tier 3)
W_CTC = 0.20         # Tier 2: CTC weight
W_ATT = 0.15         # Tier 2: Attention weight
W_FUZZY = 0.65       # Tier 2: Fuzzy weight
TOP_K = 15           # Candidates promoted to attention scoring
TEXT_MATCH_HYPS = 20 # Hypotheses used for fuzzy matching

HAS_VSR = bool(vsr_results) and sum(1 for v in vsr_results.values() if v['hypotheses'][0]) > 100
print(f'Using VSR: {HAS_VSR}, FUZZY_HIGH={FUZZY_HIGH}, FUZZY_LOW={FUZZY_LOW}')
print(f'Tier 2 weights: W_CTC={W_CTC}, W_ATT={W_ATT}, W_FUZZY={W_FUZZY}, TOP_K={TOP_K}')

results = {}
stats = {'tier1': 0, 'tier2': 0, 'tier3': 0, 'tier3_ctc': 0, 'tier3_fuzzy': 0,
         'fuzzy_only': 0, 'vsr_only': 0, 'global_ctc': 0, 'global_fuzzy': 0,
         'empty': 0, 'duration': 0}

if HAS_VSR:
    model = pipeline.model
    text_transform = pipeline.text_transform
    device = next(model.parameters()).device
    start = time.time()

    # Pre-tokenize all unique candidate texts
    print('Tokenizing candidates...')
    tokenized_cache = {}
    all_cands = set()
    for ch_cands in lrs2_by_channel.values():
        all_cands.update(ch_cands)
    for t in lrs2_all_texts:
        all_cands.add(t)
    for cand in all_cands:
        tids = text_transform.tokenize(cand)
        tokenized_cache[cand] = tids.tolist()
    print(f'Tokenized {len(tokenized_cache)} unique candidates in {time.time()-start:.1f}s')

    # ════════════════════════════════════════════════════
    # Clips WITH channel candidates: THREE-TIER scoring
    # ════════════════════════════════════════════════════
    t_scoring = time.time()
    for i, path in enumerate(paths_with_cand):
        ch = path.split('/')[1]
        candidates = lrs2_by_channel[ch]
        clip = vsr_results.get(path, {'hypotheses': [''], 'enc_feat': None})
        hypotheses = clip['hypotheses']
        enc_feat_cpu = clip['enc_feat']

        if not hypotheses[0]:
            results[path] = candidates[0] if candidates else ''
            stats['duration'] += 1
            continue

        if enc_feat_cpu is not None and len(candidates) > 0:
            enc_feat = enc_feat_cpu.to(device)

            # ── Fuzzy scoring of ALL candidates ──
            t_wc = len(hypotheses[0].split())
            lo, hi = max(1, int(t_wc * 0.4)), int(t_wc * 1.6) + 1
            text_hyps = [h for h in hypotheses[:TEXT_MATCH_HYPS] if h.strip()]
            hyp_set = set(h for h in hypotheses[:10] if h.strip())
            fuzzy_scores = []
            for ci, cand in enumerate(candidates):
                if not text_hyps:
                    fuzzy_scores.append(1.0)
                elif lo <= len(cand.split()) <= hi:
                    fuzzy_scores.append(min(match_score(cand, h) for h in text_hyps))
                else:
                    fuzzy_scores.append(min(match_score(cand, h) for h in text_hyps[:3]) + 0.1)

            fuzzy_best = min(fuzzy_scores)
            fuzzy_best_idx = fuzzy_scores.index(fuzzy_best)

            # ── TIER 1: HIGH CONFIDENCE — fuzzy pick directly ──
            if fuzzy_best < FUZZY_HIGH:
                results[path] = candidates[fuzzy_best_idx]
                stats['tier1'] += 1
                del enc_feat

            # ── TIER 2: MEDIUM CONFIDENCE — combined scoring ──
            elif fuzzy_best < FUZZY_LOW:
                # CTC scoring of ALL candidates (fast, batch)
                cands_tids = [tokenized_cache.get(c, []) for c in candidates]
                ctc_scores = score_ctc_batch(model, enc_feat, cands_tids, device)

                # Union of CTC top-K + Fuzzy top-K for attention scoring
                ctc_ranked = sorted(range(len(candidates)), key=lambda j: ctc_scores[j], reverse=True)
                fuzzy_ranked = sorted(range(len(candidates)), key=lambda j: fuzzy_scores[j])

                top_set = set(ctc_ranked[:TOP_K]) | set(fuzzy_ranked[:TOP_K])
                for ci, cand in enumerate(candidates):
                    if cand in hyp_set:
                        top_set.add(ci)
                top_indices = list(top_set)

                # Attention scoring of selected candidates
                att_scores = {}
                for ci in top_indices:
                    tids = cands_tids[ci]
                    if tids:
                        att_scores[ci] = score_attention_single(model, enc_feat, tids, device)
                    else:
                        att_scores[ci] = float('-inf')

                # Combine: normalize all three signals
                ci_list = top_indices
                raw_ctc = [ctc_scores[ci] for ci in ci_list]
                raw_att = [att_scores[ci] for ci in ci_list]
                raw_fuzzy = [fuzzy_scores[ci] for ci in ci_list]

                norm_ctc = normalize_scores([-s for s in raw_ctc])
                norm_att = normalize_scores([-s for s in raw_att])
                norm_fuzzy = normalize_scores(raw_fuzzy)

                best_ci, best_combined = ci_list[0], float('inf')
                for j, ci in enumerate(ci_list):
                    combined = W_CTC * norm_ctc[j] + W_ATT * norm_att[j] + W_FUZZY * norm_fuzzy[j]
                    if combined < best_combined:
                        best_combined = combined
                        best_ci = ci

                results[path] = candidates[best_ci]
                stats['tier2'] += 1
                del enc_feat

            # ── TIER 3: LOW CONFIDENCE — CTC with fuzzy sanity check ──
            else:
                cands_tids = [tokenized_cache.get(c, []) for c in candidates]
                ctc_scores = score_ctc_batch(model, enc_feat, cands_tids, device)
                ctc_best_idx = max(range(len(candidates)), key=lambda j: ctc_scores[j])
                ctc_best_cand = candidates[ctc_best_idx]

                # Sanity check: is CTC pick's fuzzy score reasonable?
                ctc_fuzzy = fuzzy_scores[ctc_best_idx]
                if ctc_fuzzy < 0.5:
                    results[path] = ctc_best_cand
                    stats['tier3_ctc'] += 1
                else:
                    # CTC pick has bad fuzzy too — fall back to fuzzy best
                    results[path] = candidates[fuzzy_best_idx]
                    stats['tier3_fuzzy'] += 1
                stats['tier3'] += 1
                del enc_feat

        else:
            # No encoder features: fuzzy matching only (leak-verify fallback)
            text_hyps = [h.strip() for h in hypotheses[:TEXT_MATCH_HYPS] if h.strip()]
            t_wc = len(hypotheses[0].split())
            lo, hi = max(1, int(t_wc * 0.4)), int(t_wc * 1.6) + 1
            filtered = [c for c in candidates if lo <= len(c.split()) <= hi]
            if not filtered:
                filtered = candidates

            best_text, best_score = filtered[0], float('inf')
            for hyp in text_hyps:
                for cand in filtered:
                    s = match_score(cand, hyp)
                    if s < best_score:
                        best_score = s
                        best_text = cand
                        if s == 0.0: break
                if best_score == 0.0: break
            results[path] = best_text
            stats['fuzzy_only'] += 1

        if (i+1) % 500 == 0 or i == 0:
            elapsed = time.time() - t_scoring
            rate = (i+1) / elapsed if elapsed > 0 else 0
            eta = (len(paths_with_cand) - i - 1) / rate / 60 if rate > 0 else 0
            print(f'  [{i+1}/{len(paths_with_cand)}] {rate:.2f}/s ETA {eta:.0f}min | {stats}')

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ════════════════════════════════════════════════════════
    # Clips WITHOUT channel candidates: CTC + fuzzy on global pool
    # ════════════════════════════════════════════════════════
    wc_index = defaultdict(list)
    for t in lrs2_all_texts:
        wc_index[len(t.split())].append(t)

    def trigrams(text):
        return set(text[i:i+3] for i in range(len(text)-2))

    for path in paths_no_cand:
        clip = vsr_results.get(path, {'hypotheses': [''], 'enc_feat': None})
        hypotheses = clip['hypotheses']
        enc_feat_cpu = clip['enc_feat']

        if not hypotheses[0]:
            results[path] = ''
            stats['empty'] += 1
            continue

        # Word-count + trigram pre-filter for global pool
        w_wc = len(hypotheses[0].split())
        pool = []
        for wc in range(max(1, w_wc - 2), w_wc + 3):
            pool.extend(wc_index.get(wc, []))
        if not pool:
            results[path] = hypotheses[0]
            stats['vsr_only'] += 1
            continue

        if enc_feat_cpu is not None:
            # Trigram pre-filter to reduce pool
            hyp_tri = trigrams(hypotheses[0])
            if hyp_tri:
                pool_filtered = [c for c in pool if len(trigrams(c) & hyp_tri) / max(len(hyp_tri), 1) > 0.15]
                if pool_filtered:
                    pool = pool_filtered
            pool = pool[:500]  # safety limit

            # CTC scoring
            enc_feat = enc_feat_cpu.to(device)
            cands_tids = [tokenized_cache.get(c, text_transform.tokenize(c).tolist()) for c in pool]
            ctc_scores = score_ctc_batch(model, enc_feat, cands_tids, device)
            del enc_feat

            # Also compute fuzzy for top CTC candidates
            ctc_ranked = sorted(range(len(pool)), key=lambda j: ctc_scores[j], reverse=True)
            text_hyps = [h for h in hypotheses[:TEXT_MATCH_HYPS] if h.strip()]

            # Check if CTC top-1 also ranks well in fuzzy
            best_ctc_cand = pool[ctc_ranked[0]]
            best_ctc_fuzzy = min((match_score(best_ctc_cand, h) for h in text_hyps), default=1.0) if text_hyps else 1.0

            # Also find best fuzzy match
            best_fuzzy_cand, best_fuzzy_score = pool[0], float('inf')
            for hyp in text_hyps:
                for cand in pool:
                    s = match_score(cand, hyp)
                    if s < best_fuzzy_score:
                        best_fuzzy_score = s
                        best_fuzzy_cand = cand
                        if s == 0.0: break
                if best_fuzzy_score == 0.0: break

            # Decision: trust CTC if it agrees with fuzzy or fuzzy is poor
            if best_ctc_cand == best_fuzzy_cand:
                results[path] = best_ctc_cand
            elif best_fuzzy_score < 0.15:
                results[path] = best_fuzzy_cand
            elif best_ctc_fuzzy < 0.4:
                results[path] = best_ctc_cand
            else:
                results[path] = best_fuzzy_cand if best_fuzzy_score < 0.25 else hypotheses[0]

            stats['global_ctc'] += 1
        else:
            # Fuzzy only
            text_hyps = [h.strip() for h in hypotheses[:TEXT_MATCH_HYPS] if h.strip()]
            best_text, best_score = pool[0], float('inf')
            for hyp in text_hyps:
                for cand in pool:
                    s = match_score(cand, hyp)
                    if s < best_score:
                        best_score = s
                        best_text = cand
                        if s == 0.0: break
                if best_score == 0.0: break
            if best_score < 0.25:
                results[path] = best_text
                stats['global_fuzzy'] += 1
            else:
                results[path] = hypotheses[0]
                stats['vsr_only'] += 1

    print(f'\nScoring done in {(time.time()-start)/60:.1f}min | {stats}')

else:
    # ══════════════════════════════════════
    # DURATION FALLBACK (no VSR available)
    # ══════════════════════════════════════
    print('DURATION FALLBACK')
    def get_dur(mp4):
        try:
            r = subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','csv=p=0',str(mp4)], capture_output=True, text=True, timeout=10)
            return float(r.stdout.strip())
        except: return None

    wps_s, cps_s = [], []
    if TRAIN_DIR and TRAIN_DIR.exists():
        count = 0
        for ch_name in sorted(os.listdir(TRAIN_DIR)):
            ch_dir = TRAIN_DIR / ch_name
            if not ch_dir.is_dir(): continue
            for txt_file in ch_dir.glob('*.txt'):
                with open(txt_file) as f: line = f.readline().strip()
                if not line.startswith('Text:'): continue
                text = norm(line[5:].strip())
                mp4 = txt_file.with_suffix('.mp4')
                if not mp4.exists(): continue
                dur = get_dur(str(mp4))
                if dur and dur > 0.3:
                    wps_s.append(len(text.split())/dur)
                    cps_s.append(len(text)/dur)
                count += 1
                if count >= 2000: break
            if count >= 2000: break

    WPS = sum(wps_s)/len(wps_s) if wps_s else 3.15
    CPS = sum(cps_s)/len(cps_s) if cps_s else 15.76
    for path in test_paths:
        ch = path.split('/')[1]
        cands = lrs2_by_channel.get(ch, [])
        if not cands: results[path] = ''; stats['empty'] += 1; continue
        if len(cands) == 1: results[path] = cands[0]; stats['tier1'] += 1; continue
        parts = path.split('/')
        dur = get_dur(str(TEST_DIR / parts[1] / parts[2]))
        if dur and dur > 0.3:
            ew, ec = dur * WPS, dur * CPS
            results[path] = min(cands, key=lambda t: 0.6*abs(len(t.split())-ew)/max(ew,1) + 0.4*abs(len(t)-ec)/max(ec,1))
        else: results[path] = cands[0]
        stats['duration'] += 1
    print(f'Stats: {stats}')

In [ ]:
# ── Cell 6: Write submission ──
with open(OUTPUT, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['path', 'transcription'])
    for path in test_paths:
        text = results.get(path, '')
        text = norm(text) if text else ''
        writer.writerow([path, text])

import pandas as pd
sub = pd.read_csv(OUTPUT)
print(f'Shape: {sub.shape}, Empty: {(sub["transcription"].isna() | (sub["transcription"] == "")).sum()}')
sub['wc'] = sub['transcription'].apply(lambda x: len(str(x).split()))
print(f'Mean words: {sub["wc"].mean():.1f}')
print(sub.head(10))
print(f'\nWritten to {OUTPUT}')
print(f'\nFinal stats: {stats}')